In [ ]:
from dotenv import load_dotenv
load_dotenv()

from typing_extensions import TypedDict, Annotated
from typing import List
import os
import re  #x regexs
import operator #per operatori matematici, x fare tipo operator.add(2,3) 

from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

from langchain_community.utilities import SQLDatabase  #supporta SQLite/PostgreSQL/MySQL/MSSQL/ect..

LLM_MODEL = "gpt-oss"
BASE_URL = "http://localhost:11434"

llm = ChatOllama(
    model=LLM_MODEL, 
    base_url=BASE_URL, 
    reasoning=True) #dice al modello "usa le capacità reasoning del modello" quindi SE IL MODELLO LO SUPPORTA, invece di solo ricevere l'output finale, puoi ottenere anche e.g.
    # Thinking...
    # Step 1...
    # Step 2...

response = llm.invoke("Hello, how are you?")
response.pretty_print()

In [ ]:

response.additional_kwargs['reasoning_content']  #additional_kwargs contiene metadati extra (e.g.'reasoning_content') di risposta, MA NON TUTTI I MODELS LO SUPPORTANO
db = SQLDatabase.from_uri('sqlite:///db/employees_db-full-1.0.6.db')  #connessione al db, from_uri() è factory method, sqlite:/// significa usa sqlite
tables = db.get_usable_table_names()  #recupera liste tabelle db
SCHEMA = db.get_table_info()  #estrae schema sql COMPLETO (tutte le tabs ect..)

In [ ]:
@tool
def get_database_schema(table_name: str = None):
    """Get database schema information for SQL query generation.
    Use this first to understand table structure before creating queries."""
    if table_name:
        tables = db.get_usable_table_names()  #recupera liste tabelle db
        if table_name.lower() in [t.lower() for t in tables]:
            result = db.get_table_info([table_name]) #get db schema of tab target!
            return result
        else:
            return f"Error: Table '{table_name}' not found. Available tables: '{', '.join(tables)}'"
    else:
        return SCHEMA
    
@tool
def generate_sql_query(question: str, schema_info: str=None):
    """Generate a SQL SELECT query from a natural language question using database schema.
        Always use this after getting schema information."""
    schema_to_use = schema_info if schema_info else SCHEMA
    prompt = f"""Based on this database schema:
                {schema_to_use}
                Generate a SQL query to answer this question: {question}
                Rules:
                - Use only SELECT statements
                - Include only existing columns and tables
                - Add appropriate WHERE, GROUP BY, ORDER BY clauses as needed
                - Limit results to 10 rows unless specified otherwise
                - Use proper SQL syntax for SQLite
                Return only the SQL query, nothing else.
                """
    response = llm.invoke(prompt)
    sql_query = response.content.strip()
    print(f"[TOOL] Generated SQL Query: {sql_query[:30]}...")
    return sql_query

generate_sql_query.invoke("how many employees are there?")

@tool
def validate_sql_query(query: str):
    """Validate SQL query for safety and syntax before execution.
        Returns 'Valid: <query>' if safe or 'Error: <message>' if unsafe."""
    clean_query = query.strip()
    # remove sql code block
    clean_query = re.sub(r'```sql\s*', '', clean_query, flags=re.IGNORECASE)  #rimuove il text  ```sql   dalla response
    clean_query = re.sub(r'```\s*', '', clean_query, flags=re.IGNORECASE) #rimuove il text  ```   dalla response
    clean_query = clean_query.strip().rstrip(';')  #rimuove text ';' 
    if not clean_query.lower().startswith('select'):  #blocca update/delete/drop
        return "Error: only 'select' statements are allowed."
    # Check 2: Block dangerous SQL keywords
    dangerous_keywords = ['INSERT', 'UPDATE', 'DELETE', 'ALTER', 'DROP', 'CREATE', 'TRUNCATE']
    query_upper = clean_query
    for keyword in dangerous_keywords:
        if keyword in query_upper:
            return f"Error: {keyword} operations are not allowed."
    print("[TOOL] Your sql query is validated. Passed!")
    return clean_query

validate_sql_query.invoke('SELECT COUNT(*) FROM employees;')

In [ ]:
@tool
def execute_sql_query(sql_query: str):
    """Execute a validated SQL query and return results.
    Only use this after validating the query for safety."""
    query = validate_sql_query.invoke(sql_query)
    if query.startswith('Error:'):
        return f"Query '{sql_query}' validation failed with Error: {query}"
    result = db.run(query)
    if result:
        return  f"Query Results: {result}"
    else:
        return f"Query Executed Sucessfully but No Result was Found!"
    
execute_sql_query.invoke('SELECT COUNT(*) FROM employees;')

@tool
def fix_sql_error(original_query: str, error_message: str, question):
    """Fix a failed SQL query by analyzing the error and generating a corrected version.
        Use this when validation or execution fails."""
    fix_prompt = f"""The following SQL query failed:
        Query: {original_query}
        Error: {error_message}
        Original Question: {question}
        Database Schema:
        {SCHEMA}
        Analyze the error and provide a corrected SQL query that:
        1. Fixes the specific error mentioned
        2. Still answers the original question
        3. Uses only valid table and column names from the schema
        4. Follows SQLite syntax rules
        Return only the corrected SQL query, nothing else.
        """
    response = llm.invoke(fix_prompt)
    query = response.content.strip()
    print(f"[TOOL] Generated fixed SQL Query.")
    return query

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[list, operator.add]

tools = [
    get_database_schema,  #ur custom fuction marchiata come @tool
    generate_sql_query,
    execute_sql_query,
    fix_sql_error
]

llm_with_tools = llm.bind_tools(tools)
llm_with_tools

In [ ]:
def agent_node(state: AgentState):
    system_prompt = f"""You are an expert SQL analyst working with an employees database.
    Database Schema:
    {SCHEMA}
    Your workflow for answering questions:
    1. Use `get_database_schema` first to understand available tables and columns (if needed)
    2. Use `generate_sql_query` to create SQL based on the question
    3. Use `execute_sql_query` to run the validated query
    4. If there's an error, use `fix_sql_error` to correct it and try again (up to 3 times)
    Rules:
    - Always follow the workflow step by step
    - If a query fails, use the fix tool and try again
    - Provide clear, informative answers
    - Be precise with table and column names
    - Handle errors gracefully and try to fix them
    - If you fail after 3 attempts, explain what went wrong
    Available tools:
    - get_database_schema: Get table structure info
    - generate_sql_query: Create SQL from question
    - execute_sql_query: Run the query
    - fix_sql_error: Fix failed queries
    Remember: Always validate queries before executing them for safety.
    """
    messages = [SystemMessage(system_prompt)] + state['messages']
    response = llm_with_tools.invoke(messages)
    return {'messages': [response]}  #update stato condiviso

llm_with_tools.invoke('how many employees are there?')

In [ ]:
def should_continue(state: AgentState):
    last_message = state['messages'][-1]
    if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
        print("[TOOL] Calling the tool")
        for tc in last_message.tool_calls:
            print(f"Callling tool: '{tc['name']}' with Args: '{tc['args']}'")
        return 'tools'
    else:
        print("[AGENT] Agent is processing your request...")
        return END
    
def create_sql_agent():
    builder = StateGraph(AgentState)
    # add nodes
    builder.add_node('agent', agent_node)
    builder.add_node('tools', ToolNode(tools))
    # add edges
    builder.add_edge(START, 'agent')
    builder.add_edge('tools', 'agent')
    # add router
    builder.add_conditional_edges('agent', should_continue, ['tools', END])
    graph = builder.compile()
    return graph

agent = create_sql_agent()
agent

In [ ]:

query = "how many employees are there"
result = agent.invoke({'messages': [query]})

db.run('SELECT COUNT(*) FROM employees;')  #execute

result['messages'][-1].pretty_print()

query = "What is the average salary of each department show me top 5 department?"
result = agent.invoke({'messages': [query]})
result['messages'][-1].pretty_print()

query = "Show me the top 5 highest paid employees with their title and salaries?"
result = agent.invoke({'messages': [query]})
result['messages'][-1].pretty_print()